# 摘要:

伴随着基于物理的渲染在实时渲染领域广泛使用，区域光在渲染中的重要性也变得越来越强。对一些特定类型的区域光源以及BRDF，存在一个数学上的解析解，然后这些解析解并不能使用在Microfacet BRDF上，目前存在的一些求解方向包括Most Representative Point, Linearly Transform Cosine (LTC), Analytic spherical harmonic coefficients for polygonal area lights等。其中LTC由于其精确性，较好的性能，多种光源类型的支持成为游戏等实时渲染应用程序的首选方案。
然而要将LTC在移动平台的生产项目中使用仍然存在不少挑战。例如horizon clipping 问题，三角形与平面求交从而得到新的多边形，然后在shader里实现这样的函数会造成大量的代码分支以及寄存器的占用。还有区域光的裁切问题，如何能够找到一个区域光的几何代理用来与模型进行求交计算？如何在tiled/clustered renderer上对区域光做binning？另外还有移动平台上浮点数计算精度的问题，在移动平台上大量的计算将在FP16上进行，然而从之前的实践中我们得知LTC对计算精度有着很高的要求。还有区域光如果做Level of detail管理？我们将对这一系列在实践中将会问题的展开讨论。


# Polygonal integration over Lambert surface

多边形在半球面的cosine分布函数下，存在解析解:
$$
E(p_1, ..., p_n) = \frac 1 {2 \pi} \sum_{i=1}^{n} acos(\langle p_i, p_j \rangle) \langle {\frac {p_i \times p_j} {\| {p_i \times p_j} \|} }, {\begin{bmatrix} 0\\ 0\\ 1\\ \end{bmatrix}} \rangle \quad \textrm{where} \quad j = i + 1
$$

原论文中仅给出公式，这里对这个公式给出具体推导。推导主要是使用了[斯托克斯定理](https://en.wikipedia.org/wiki/Stokes%27_theorem)。
$$
\oint _{\Gamma }\mathbf {F} \,\cdot \,d{\mathbf {\Gamma } }=\iint _{S}\nabla \times \mathbf {F} \,\cdot \,d\mathbf {S} 
$$

首先我们要求解的问题为多边形在球面cosine函数下的积分
$$
{\frac 1 \pi} {\int_{P} cos(\overrightarrow{\rm n},\overrightarrow{\rm \omega}) } d\omega
$$


# LUT fitting and normalization

最终生成的LTC矩阵存在5个变量，很难直接求解这个5维的优化问题，我们要想办法降低问题的复杂性，首先我们先尝试将GGX替换成Phong模型，在Phong模型的情况下，可以将cosine lobe通过旋转与缩放来达到对phong的拟合，一个LTC矩阵可以分解为旋转与缩放矩阵，其中旋转矩阵的Y轴为反射向量，由于Phong模型是各项同性的，所以缩放矩阵的m00与m11相等，问题简化为单变量的优化问题，就很容易解决了。下图为两个矩阵变换得直观显示。


有了上面的经验之后，我们再尝试将Phong模型替换成GGX。下图为Phong与GGX lobe的对比:

从上面我们可以观察到，GGX为各项异性且lobe的主要方向与反射向量存在一定夹角。我们继续按照将矩阵分解成旋转与缩放矩阵的思路进行求解，其中旋转矩阵与Phong模型一致。此时通过我们观察到的GGX对比Phong的性质，可以通过m00与m11来对X,Y方向的缩放以及m20对YZ平面上的旋转来达到GGX的近似，具体效果见下图：

最终我们通过将LTC变换矩阵分解成旋转矩阵以及本地变换矩阵（X,Y方向的缩放以及YZ平面上的旋转），将一个5维的优化问题简化成3维的优化问题。

TODO: LUT fitting implementation

最终生成的LUT需要存储5个变量，这样需要两次贴图查找操作，我们希望将变量控制在4个以内，这样只需要一次查表操作。让我们回顾一下LTC中关于D的解析式
$$
D(\omega) = D_o({\omega}_o) {\frac {\partial {\omega}_o} {\partial \omega}} 
          = D_o (\frac {M^{-1} \omega} {\| {M^{-1} \omega} \|}) {\frac {| M^{-1} |} {{\| {|M^{-1} \omega} \|}^3}}
$$

其中$M^{-1}$为我们生成的LUT表。通过上的解析式分析我们可以得到，将$\lambda I M^{-1}$替换掉$M^{-1}$，上面表达式依然成立, 其中$I$为单位矩阵。
$$
D(\omega) = D_o (\frac {\lambda I  M^{-1} \omega} {\| \lambda I {M^{-1} \omega} \|}) {\frac {| \lambda I M^{-1} |} {{\| \lambda I {M^{-1} \omega} \|}^3}}  \\
          = D_o (\frac {M^{-1} \omega} {\| {M^{-1} \omega} \|}) {\frac {{| \lambda I|} {| M^{-1} |}} { {{\lambda}^3} {{\| {M^{-1} \omega} \|}^3}} } \\
          = D_o (\frac {M^{-1} \omega} {\| {M^{-1} \omega} \|}) {\frac {{{\lambda}^3} {| M^{-1} |}} { {{\lambda}^3} {{\| {M^{-1} \omega} \|}^3}} }
$$

所以我们可以将LUT矩阵除以m00, m11, m22任意一个来达到将数据压缩成4个的目的。下图为分别使用m00, m11, m22来进行归一化后的数据展示。

# Naïve implementation


# Horizon clipping

第一章节中提到的多边形在lambert表面的积分是对clamped cosine函数进行求积，所以我们需要将多边形裁切到上半球，shader里实现多边形的裁切会造成大量的代码分支，以下为伪代码示例。

```
    /* Detect clipping config */
    int config = 0;
    if(L[0].z > 0.0) config += 1;
    if(L[1].z > 0.0) config += 2;
    if(L[2].z > 0.0) config += 4;
    if(L[3].z > 0.0) config += 8;

    if(config == 0) {
        // clip all
    } else if(config == 1) { // V1 clip V2 V3 V4
        ...
    } else if(config == 2) { // V2 clip V1 V3 V4
        ...
    ...
    } else if(config == 14) { // V2 V3 V4 clip V1
        ...
    } else if(config == 15) { // V1 V2 V3 V4
        n = 4;
    }
```

原论文并没有对裁切这部分的实践做介绍，不过Stephen Hill在[Real-Time Area Lighting:a Journey from Research to Production](https://advances.realtimerendering.com/s2016/s2016_ltc_rnd.pdf)中给出了一个近似方案，这里做一下简单的介绍。

TODO: theory explanation

TODO: LUT polynomial fit

由于使用球来近似平面，无法处理平面的朝向问题，所以我们需要自己来处理平面的朝向，将当前位置带入平面方程就可以得到当前位置处于 平面的正面或者方面，通过对F向量取反即可处理双面光源。

# Geometry proxy (TODO)